In [5]:
from typing import Optional, Union
import time
import json
import requests

class RateLimiter:
    def __init__(self, max_calls: int = 4, time_window: float = 1.0):
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls = []

    def wait(self):
        now = time.time()
        self.calls = [t for t in self.calls if now - t < self.time_window]
        if len(self.calls) >= self.max_calls:
            sleep_for = self.time_window - (now - self.calls[0])
            if sleep_for > 0:
                time.sleep(sleep_for)
        self.calls.append(time.time())


class SECAPIClient:
    """
    - SEC 권장: 적절한 User-Agent(이메일 포함), 과도한 호출 금지.
    - Company Facts는 CIK(10자리 zero-pad)가 필요합니다.
    """
    def __init__(self, user_agent: str, rate_limiter: RateLimiter,
                 base_url: str = "https://data.sec.gov"):
        if not user_agent or "@" not in user_agent:
            raise ValueError("SEC User-Agent는 이메일 주소를 포함해야 합니다.")
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Accept": "application/json"
        })
        self.rl = rate_limiter
        self._ticker_cik_cache = None  # lazy load

    # --- 내부 유틸 ---
    def _get(self, path: str, params: Optional[dict] = None,
             max_retries: int = 3, backoff: float = 1.0):
        url = f"{self.base_url}{path}"
        last_err = None
        for attempt in range(1, max_retries + 1):
            self.rl.wait()
            try:
                resp = self.session.get(url, params=params, timeout=30)
                if resp.status_code == 200:
                    if resp.text.strip() == "":
                        return {}
                    return resp.json()
                elif resp.status_code in (429, 503):
                    time.sleep(backoff * attempt)
                    continue
                else:
                    raise RuntimeError(f"HTTP {resp.status_code} for {url}: {resp.text[:200]}")
            except requests.RequestException as e:
                last_err = e
                time.sleep(backoff * attempt)

        raise RuntimeError(f"GET 실패: {url} (retries={max_retries}) | last_err={last_err}")

    @staticmethod
    def _to_cik10(cik: Union[int, str]) -> str:
        return f"{int(cik):010d}"

    # --- 티커 → CIK 매핑 ---
    def _load_ticker_map(self):
        if self._ticker_cik_cache is not None:
            return

        data = self._get("/files/company_tickers.json")
        mapping = {}
        for _, rec in data.items():
            tkr = str(rec.get("ticker", "")).upper().strip()
            cik = rec.get("cik_str")
            if tkr and cik:
                mapping[tkr] = self._to_cik10(cik)
        if not mapping:
            raise RuntimeError("company_tickers.json 로드에 실패했습니다 (매핑 비어있음).")
        self._ticker_cik_cache = mapping

    def get_cik_by_ticker(self, ticker: str) -> str:
        self._load_ticker_map()
        t = ticker.upper().strip()
        cik = self._ticker_cik_cache.get(t)
        if not cik:
            raise ValueError(f"미지원/미등록 티커: {ticker}")
        return cik

    def get_company_facts_by_cik(self, cik10: str) -> dict:
        path = f"/api/xbrl/companyfacts/CIK{cik10}.json"
        return self._get(path)

    def get_company_facts_by_ticker(self, ticker: str) -> dict:
        cik10 = self.get_cik_by_ticker(ticker)
        return self.get_company_facts_by_cik(cik10)


In [8]:
client.get_cik_by_ticker("AAPL")


AttributeError: 'SECAPIClient' object has no attribute 'get_cik_by_ticker'